In [ ]:
# Allow importing from src
import sys
sys.path.insert(0, '../src/')

# Fix for draw_geometries crashing on Wayland
import os
os.environ["XDG_SESSION_TYPE"] = "x11"

In [ ]:
import open3d as o3d
from pathlib import Path
import torch
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from torchvision.utils import make_grid
import optuna
from copy import deepcopy
from tqdm.notebook import tqdm

from ingp import LInstantNGP
from utils.rays import create_intrinsic, create_rays, look_at, equidistance_rotations, sample_ray_uniformally, render_rays
from utils.data import dbscan_and_connected_merge, find_rmn_angles
from utils.lutils import NeRFData

COMPUTE_DEVICE = torch.device('cpu')
if torch.cuda.is_available():
    COMPUTE_DEVICE = torch.device('cuda:0')
elif torch.mps.is_available():
    COMPUTE_DEVICE = torch.device('mps')
print(f"{COMPUTE_DEVICE=}")

RUN_OPT = False

In [ ]:
print("Available GSO objects:")
print("\n".join([p.stem for p in Path(f'../data/GSO/').iterdir() if p.is_dir()]))

# NeRF to point cloud evaluation

## Setup

In [ ]:
def get_origin_direction_c2w_intrinsic(img_shape, c2ws, intrinsics):
    origin, direction = [], []
    for c2w, intrinsic in zip(c2ws, intrinsics):
        o, d = create_rays(img_shape[0], img_shape[1], intrinsic, c2w)
        origin.append(o)
        direction.append(d)

    return torch.stack(origin), torch.stack(direction)

In [ ]:
MODEL_CHKPT_PATH = Path(
    "../lightning_logs/ingp_DTU_scan37_m_ds/checkpoints/best_val_psnr_epoch=13.ckpt"
).resolve()

hparams_path = (MODEL_CHKPT_PATH / ".." / ".." / "hparams.yaml").resolve()

model = LInstantNGP.load_from_checkpoint(MODEL_CHKPT_PATH, map_location=COMPUTE_DEVICE, hparams_file=hparams_path)
model.freeze()
model.eval()

data = NeRFData.load_from_checkpoint(MODEL_CHKPT_PATH, map_location=torch.device('cpu'), hparams_file=hparams_path)
data._set_hparams(model.hparams)
data.setup("predict")

idxs = find_rmn_angles(data.c2ws, angle_count=8)
origin, direction = get_origin_direction_c2w_intrinsic((1200, 1600), data.c2ws[idxs], data.intrinsics[idxs])
print(f"{origin.shape=} {direction.shape=}")

# Depth cloud

In [ ]:
def cloud_from_tensor(tens):
    return o3d.geometry.PointCloud(o3d.utility.Vector3dVector(tens.cpu()))

In [ ]:
alpha_mask = data.images[idxs, ..., -1] != 0
dl = DataLoader(TensorDataset(origin[alpha_mask], direction[alpha_mask]), batch_size=2**9)

rm_depth = []
for o, di in tqdm(dl, total=len(dl), unit="batch", postfix="batch_size=2^9"):
    o, di = o.to(model.device), di.to(model.device)
    _, de, acc = model.render_rays(o, di)
    de = de.unsqueeze(-1)
    points = o + de * di
    mask = model.nerf(points, None, skip_colors=True) < model.hparams.f_sigma_threshold

    near_plane = torch.sqrt(torch.sum(torch.pow(o, 2), -1, keepdim=True)) + model.near_offset
    de[mask | (acc < 0.99) | (de < near_plane)] = torch.inf

    rm_depth.append(de.cpu())
    if model.device == torch.device("cuda:0"):
        torch.cuda.empty_cache()

rm_depth = torch.cat(rm_depth, 0)

In [ ]:
img = torch.full_like(alpha_mask[0], torch.inf, dtype=torch.float32)
img[alpha_mask[0]] = rm_depth.squeeze(-1)

img.shape

In [ ]:
fix, ax = plt.subplots(1,1, figsize=(20, 20))
ax.imshow((img.unsqueeze(-1).expand(-1, -1, 3) - 3.4002 - model.near_offset) / (model.far_offset - model.near_offset))

In [ ]:
mask = (rm_depth != torch.inf).squeeze(-1)

rm_points = origin[alpha_mask][mask] + rm_depth[mask] * direction[alpha_mask][mask]

bbox_mask = (rm_points.abs() <= 1.0).all(-1)
rm_points= rm_points[bbox_mask]

rm_points.shape

In [ ]:
point_cloud = cloud_from_tensor(rm_points)
point_cloud

In [ ]:
normals = []
normal_points = DataLoader(
    torch.from_numpy(np.asarray(point_cloud.points, dtype=np.float32)),
    batch_size=2**19, shuffle=False
)
for nps in normal_points:
    normals.append(model.estimate_normals(nps).cpu())
normals = torch.cat(normals, dim=0)
point_cloud.normals = o3d.utility.Vector3dVector(normals)

In [ ]:
pc = point_cloud.voxel_down_sample(0.002)
pc

In [ ]:
pc.estimate_normals(o3d.geometry.KDTreeSearchParamKNN(100))

In [ ]:
pc.paint_uniform_color([0.5, 0.5, 0.5])
o3d.visualization.draw_geometries([pc])

In [ ]:
o3d.io.write_point_cloud("rmn_cloud_raw_naive.ply", point_cloud, write_ascii=True)

In [ ]:
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pc, depth=9)
mesh.compute_vertex_normals()

vertices_to_remove = densities < np.quantile(densities, 0.1)
mesh.remove_vertices_by_mask(vertices_to_remove)

# Remove unused vertices and tidy up
mesh.remove_unreferenced_vertices()
mesh.remove_degenerate_triangles()
mesh.remove_duplicated_triangles()

o3d.visualization.draw_geometries([mesh])

In [ ]:
arr = np.load(f"../data/DTU/scan37/cameras.npz")
mesh.transform(arr["scale_mat_0"])

In [ ]:

o3d.io.write_triangle_mesh("rmn_mesh_naive.ply", mesh, write_ascii=True)